# Avaliação Final — Piloto de Cobrança Preventiva (BemLar)

Este notebook é o **esqueleto** da sua entrega. Os blocos abaixo dizem *o que* precisa ser feito; o código é seu.

Antes de escrever qualquer linha:
1. Leia `00_email_financeiro.md` inteiro.
2. Leia `dicionario.md` inteiro.
3. Abra os cinco CSVs e **olhe os valores**, não só os nomes das colunas.

> A entrega vale mais pelas decisões registradas do que pelo código. Cada bloco marcado com ❓ pede uma resposta escrita — responda em célula markdown, ali mesmo.

---
## Fase 1 — Entendimento do Negócio

❓ Em uma frase, qual é a pergunta de negócio? Qual é a unidade de análise (o que é uma linha)?

❓ O Ricardo fez 5 pedidos no e-mail. Liste os 5 e, para cada um, marque agora sua intenção: **atender**, **atender com ressalva** ou **recusar**. Você pode mudar de ideia depois — mas registre a versão inicial.

In [ ]:
# Fase 1 - espaco livre (nao precisa de codigo aqui, se nao quiser)


---
## Fase 2 — Entendimento dos Dados

Carregue os cinco arquivos. Lembre: separador `;`, decimal `,`, datas `dd/mm/aaaa`.

Para **cada** base, responda:
- Qual é a granularidade (o que é uma linha)?
- Quantos contratos ela cobre? Todos, ou só uma parte?
- O que significa um valor vazio nessa base? Sempre a mesma coisa?

❓ Faça um inventário de problemas de qualidade: nulos, valores impossíveis, categorias que deveriam ser a mesma. Registre em uma tabela: problema | onde | quantas linhas | é erro ou artefato | o que você fez.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

KW = dict(sep=";", decimal=",", encoding="utf-8")
BASE_DIR = Path("bases")
DATE_COLUMNS = {
    "contratos": ["data_venda", "data_snapshot"],
    "pagamentos": ["data_vencimento", "data_pagamento"],
    "compras": ["data_compra"],
    "ocorrencias_sac": ["data_ocorrencia"],
    "score_bureau": ["data_consulta"],
    "motivos_atraso": ["data_registro"],
}


def carregar_e_limpar_dados(caminho_dados):
    """Carrega os CSVs brutos e aplica apenas limpezas reproduzíveis."""
    caminho_dados = Path(caminho_dados)
    arquivos = {
        "contratos": "contratos.csv",
        "pagamentos": "pagamentos.csv",
        "compras": "compras.csv",
        "ocorrencias_sac": "ocorrencias_sac.csv",
        "score_bureau": "score_bureau.csv",
        "motivos_atraso": "motivos_atraso.csv",
    }

    dados = {
        nome: pd.read_csv(caminho_dados / arquivo, **KW)
        for nome, arquivo in arquivos.items()
    }

    for nome, colunas_data in DATE_COLUMNS.items():
        for coluna in colunas_data:
            dados[nome][coluna] = pd.to_datetime(
                dados[nome][coluna], format="%d/%m/%Y", errors="raise"
            )

    # Espaços em torno de categorias não carregam significado de negócio.
    for df in dados.values():
        for coluna in df.select_dtypes(include="object"):
            df[coluna] = df[coluna].str.strip()

    contratos = dados["contratos"].copy()
    contratos["flag_idade_suspeita"] = (contratos["idade"] > 90).astype("int8")
    contratos["idade_tratada"] = contratos["idade"].clip(upper=90)
    contratos["flag_renda_zero"] = (contratos["renda_declarada"] == 0).astype("int8")
    contratos["renda_declarada_tratada"] = contratos["renda_declarada"].replace(0, np.nan)
    dados["contratos"] = contratos

    # Duplicatas completas no SAC representam o mesmo atendimento extraído duas vezes.
    dados["ocorrencias_sac"] = dados["ocorrencias_sac"].drop_duplicates().copy()
    dados["ocorrencias_sac"]["tipo_ocorrencia_norm"] = (
        dados["ocorrencias_sac"]["tipo_ocorrencia"].str.casefold()
    )

    if dados["contratos"]["id_contrato"].duplicated().any():
        raise ValueError("id_contrato precisa ser único em contratos.csv")
    if dados["pagamentos"].duplicated(["id_contrato", "n_parcela"]).any():
        raise ValueError("(id_contrato, n_parcela) precisa ser único em pagamentos.csv")

    return dados


dados = carregar_e_limpar_dados(BASE_DIR)
contratos = dados["contratos"]
pagamentos = dados["pagamentos"]
compras = dados["compras"]
ocorrencias_sac = dados["ocorrencias_sac"]
score_bureau = dados["score_bureau"]
motivos_atraso = dados["motivos_atraso"]

In [2]:
# Inventário reprodutível de qualidade após a limpeza.
linhas_qualidade = []
for nome, df in dados.items():
    linhas_qualidade.append(
        {
            "base": nome,
            "linhas": len(df),
            "linhas_duplicadas": int(df.duplicated().sum()),
            "nulos_ou_vazios": int(df.isna().sum().sum()),
        }
    )

inventario_qualidade = pd.DataFrame(linhas_qualidade)
display(inventario_qualidade)

problemas_qualidade = pd.DataFrame(
    [
        ["2 duplicatas completas removidas", "ocorrencias_sac", 2, "erro de extração", "drop_duplicates antes de agregar"],
        ["Idade acima de 90", "contratos.idade", int(contratos["flag_idade_suspeita"].sum()), "valor improvável", "flag + limite superior de 90"],
        ["Renda declarada igual a zero", "contratos.renda_declarada", int(contratos["flag_renda_zero"].sum()), "não permite calcular razão", "flag; converter para ausente no cálculo"],
        ["Pagamento vazio", "pagamentos.data_pagamento", int(pagamentos["data_pagamento"].isna().sum()), "artefato esperado", "representar como não pago na data de referência"],
        ["Motivos sem chave de junção", "motivos_atraso", len(motivos_atraso), "limitação da fonte", "não usar em features"],
    ],
    columns=["problema", "onde", "quantidade", "classificacao", "tratamento"],
)
display(problemas_qualidade)

,base,linhas,linhas_duplicadas,nulos_ou_vazios
0,contratos,3000,0,4
1,pagamentos,28861,0,1690
2,compras,3983,0,0
3,ocorrencias_sac,3444,0,0
4,score_bureau,4490,0,0
5,motivos_atraso,1019,87,0


,problema,onde,quantidade,classificacao,tratamento
0,2 duplicatas completas removidas,ocorrencias_sac,2,erro de extração,drop_duplicates antes de agregar
1,Idade acima de 90,contratos.idade,3,valor improvável,flag + limite superior de 90
2,Renda declarada igual a zero,contratos.renda_declarada,4,não permite calcular razão,flag; converter para ausente no cálculo
3,Pagamento vazio,pagamentos.data_pagamento,1690,artefato esperado,representar como não pago na data de referência
4,Motivos sem chave de junção,motivos_atraso,1019,limitação da fonte,não usar em features


---
## Fase 3 — Preparação dos Dados

### 3.1 Construir o alvo

A regra está no enunciado e no dicionário. Você precisa, para cada contrato:
1. identificar a **parcela de referência**;
2. calcular a **data de referência**;
3. derivar `inadimplente_30d`.

❓ Depois de construir: qual é a prevalência de positivos? Ela é compatível com o que o Ricardo descreveu no e-mail?

❓ Compare o seu alvo com `status_contrato`. Eles concordam? Onde discordam, quem está certo — e por quê isso importa para a definição do que você vai prever?

In [3]:
# A parcela de referência é a última vencida até 30 dias antes do snapshot.
snapshot = contratos["data_snapshot"].iloc[0]
data_limite_referencia = snapshot - pd.Timedelta(days=30)

parcelas_elegiveis = pagamentos.loc[
    pagamentos["data_vencimento"].le(data_limite_referencia)
].sort_values(["id_contrato", "data_vencimento", "n_parcela"])

parcela_referencia = (
    parcelas_elegiveis.groupby("id_contrato", as_index=False)
    .tail(1)
    .rename(
        columns={
            "n_parcela": "n_parcela_referencia",
            "data_vencimento": "data_vencimento_referencia",
            "data_pagamento": "data_pagamento_referencia",
        }
    )
)

# O mesmo marco temporal é usado para todos os contratos: sete dias antes do vencimento.
base_referencia = contratos[["id_contrato", "id_cliente"]].merge(
    parcela_referencia[
        [
            "id_contrato",
            "n_parcela_referencia",
            "data_vencimento_referencia",
            "data_pagamento_referencia",
        ]
    ],
    on="id_contrato",
    how="inner",
    validate="one_to_one",
)
base_referencia["data_referencia"] = (
    base_referencia["data_vencimento_referencia"] - pd.Timedelta(days=7)
)

# Como toda parcela de referência já teve 30 dias para maturar no snapshot,
# pagamento vazio ou posterior a D+30 representa inadimplência de 30 dias.
limite_pagamento_30d = (
    base_referencia["data_vencimento_referencia"] + pd.Timedelta(days=30)
)
base_referencia["inadimplente_30d"] = (
    base_referencia["data_pagamento_referencia"].isna()
    | base_referencia["data_pagamento_referencia"].gt(limite_pagamento_30d)
).astype("int8")

assert base_referencia["id_contrato"].nunique() == len(contratos)
assert base_referencia["data_referencia"].lt(
    base_referencia["data_vencimento_referencia"]
).all()

print("Prevalência de inadimplência 30d:", base_referencia["inadimplente_30d"].mean().round(3))
display(base_referencia.head())

Prevalência de inadimplência 30d: 0.433


,id_contrato,id_cliente,n_parcela_referencia,data_vencimento_referencia,data_pagamento_referencia,data_referencia,inadimplente_30d
0,C00001,CLI00533,9,2026-01-29,2026-02-05,2026-01-22,0
1,C00003,CLI00116,11,2026-02-09,2026-02-05,2026-02-02,0
2,C00004,CLI00930,10,2025-09-12,NaT,2025-09-05,1
3,C00005,CLI02675,12,2025-09-10,NaT,2025-09-03,1
4,C00007,CLI00506,9,2026-02-05,NaT,2026-01-29,1


### 3.2 A função de corte temporal

Está pronta. Use em **toda** base de eventos antes de agregar qualquer coisa.

In [4]:
def filtra_por_data(df_eventos, chave, col_data, datas_ref, col_ref="data_referencia"):
    """Mantém eventos que já existiam na data de referência de cada chave."""
    out = df_eventos.merge(
        datas_ref[[chave, col_ref]], on=chave, how="inner", validate="many_to_one"
    )
    out = out.loc[out[col_data].le(out[col_ref])]
    return out.drop(columns=[col_ref])

### 3.3 Construir as features

Uma linha por contrato. Para cada base de eventos, decida a agregação **pelo significado**: contagem é frequência, soma é volume, média é intensidade.

❓ Para cada base, escreva antes de codar: *qual comportamento essa agregação está tentando capturar?*

❓ Ausência de evento vira 0, nulo, ou mediana? A resposta é a mesma para toda coluna?

Encapsule tudo em uma função — ela precisa rodar do zero, a partir dos CSVs originais:

```python
def construir_features(caminho_dados, datas_referencia):
    ...
    return df
```

In [5]:
def construir_features(caminho_dados, datas_referencia):
    """Gera uma matriz temporalmente segura: uma linha por contrato."""
    dados_locais = carregar_e_limpar_dados(caminho_dados)
    contratos_locais = dados_locais["contratos"]
    pagamentos_locais = dados_locais["pagamentos"]
    compras_locais = dados_locais["compras"]
    sac_locais = dados_locais["ocorrencias_sac"]
    bureau_locais = dados_locais["score_bureau"]

    colunas_ref = ["id_contrato", "id_cliente", "data_referencia", "inadimplente_30d"]
    referencia = datas_referencia[colunas_ref].copy()
    if referencia["id_contrato"].duplicated().any():
        raise ValueError("datas_referencia deve conter uma única linha por contrato")

    # Features disponíveis na contratação.
    features = referencia.merge(
        contratos_locais[
            [
                "id_contrato", "n_parcelas", "valor_parcela",
                "renda_declarada_tratada", "canal_venda", "data_venda",
            ]
        ],
        on="id_contrato",
        how="left",
        validate="one_to_one",
    )
    features["tenure_dias"] = (
        features["data_referencia"] - features["data_venda"]
    ).dt.days
    features["comprometimento_renda"] = (
        features["valor_parcela"] / features["renda_declarada_tratada"]
    )
    features = features.drop(columns="data_venda")

    # Quantidade de contratos parcelados do mesmo cliente já existentes na referência.
    contratos_cliente = referencia[["id_contrato", "id_cliente", "data_referencia"]].merge(
        contratos_locais[["id_contrato", "id_cliente", "data_venda"]],
        on="id_cliente",
        how="left",
        suffixes=("_alvo", "_historico"),
    )
    contratos_cliente = contratos_cliente.loc[
        contratos_cliente["data_venda"].le(contratos_cliente["data_referencia"])
    ]
    historico_contratos = (
        contratos_cliente.groupby("id_contrato_alvo", as_index=False)
        .agg(qtd_contratos_cliente_pre_ref=("id_contrato_historico", "nunique"))
        .rename(columns={"id_contrato_alvo": "id_contrato"})
    )
    historico_contratos["qtd_contratos_outros_pre_ref"] = (
        historico_contratos["qtd_contratos_cliente_pre_ref"] - 1
    )
    features = features.merge(historico_contratos, on="id_contrato", how="left")

    # Compras são eventos por cliente; por isso o filtro recebe id_contrato e id_cliente.
    compras_ref = compras_locais.merge(
        referencia[["id_contrato", "id_cliente", "data_referencia"]],
        on="id_cliente",
        how="inner",
    )
    compras_ref = compras_ref.loc[
        compras_ref["data_compra"].le(compras_ref["data_referencia"])
    ].copy()
    compras_agregadas = (
        compras_ref.groupby("id_contrato", as_index=False)
        .agg(
            qtd_compras=("valor", "size"),
            ticket_medio_compras=("valor", "mean"),
            ultima_compra=("data_compra", "max"),
        )
    )
    compras_categoria = pd.crosstab(
        compras_ref["id_contrato"], compras_ref["categoria"]
    ).reindex(columns=["Informática", "Móveis", "Eletro", "Cama/Mesa/Banho"], fill_value=0)
    compras_categoria = compras_categoria.rename(
        columns={
            "Informática": "qtd_compra_informatica",
            "Móveis": "qtd_compra_moveis",
            "Eletro": "qtd_compra_eletro",
            "Cama/Mesa/Banho": "qtd_compra_cama_mesa_banho",
        }
    ).reset_index()
    features = features.merge(compras_agregadas, on="id_contrato", how="left")
    features = features.merge(compras_categoria, on="id_contrato", how="left")
    features["dias_desde_ultima_compra"] = (
        features["data_referencia"] - features["ultima_compra"]
    ).dt.days
    features = features.drop(columns="ultima_compra")

    # Pagamentos: somente parcelas vencidas antes do instante da previsão.
    pagamentos_ref = pagamentos_locais.merge(
        referencia[["id_contrato", "data_referencia"]], on="id_contrato", how="inner"
    )
    pagamentos_ref = pagamentos_ref.loc[
        pagamentos_ref["data_vencimento"].lt(pagamentos_ref["data_referencia"])
    ].copy()
    pagamentos_ref["paga_ate_referencia"] = (
        pagamentos_ref["data_pagamento"].notna()
        & pagamentos_ref["data_pagamento"].le(pagamentos_ref["data_referencia"])
    )
    pagamentos_ref["pendente_na_referencia"] = ~pagamentos_ref["paga_ate_referencia"]
    pagamentos_ref["atraso_positivo"] = (
        (
            pagamentos_ref["paga_ate_referencia"]
            & pagamentos_ref["data_pagamento"].gt(pagamentos_ref["data_vencimento"])
        )
        | pagamentos_ref["pendente_na_referencia"]
    )
    pagamentos_agregados = (
        pagamentos_ref.groupby("id_contrato", as_index=False)
        .agg(
            qtd_parcelas_vencidas=("n_parcela", "size"),
            qtd_parcelas_pagas=("paga_ate_referencia", "sum"),
            qtd_parcelas_pendentes=("pendente_na_referencia", "sum"),
            qtd_atrasos_positivos=("atraso_positivo", "sum"),
        )
    )
    features = features.merge(pagamentos_agregados, on="id_contrato", how="left")

    # Negociação de dívida não possui nenhum evento antes da referência: seria constante.
    # Os canais abaixo representam qualquer atendimento SAC prévio, e não negociação futura.
    sac_ref = filtra_por_data(
        sac_locais, "id_contrato", "data_ocorrencia", referencia
    )
    negociacoes_pre_ref = sac_ref.loc[
        sac_ref["tipo_ocorrencia_norm"].eq("negociacao de divida")
    ]
    assert negociacoes_pre_ref.empty, "Rever o corte temporal: negociação deveria ser pós-referência neste recorte"
    sac_canais = pd.crosstab(sac_ref["id_contrato"], sac_ref["canal"]).reindex(
        columns=["Telefone", "WhatsApp", "Loja", "Site"], fill_value=0
    )
    sac_canais = sac_canais.rename(
        columns={
            "Telefone": "qtd_sac_telefone",
            "WhatsApp": "qtd_sac_whatsapp",
            "Loja": "qtd_sac_loja",
            "Site": "qtd_sac_site",
        }
    ).reset_index()
    features = features.merge(sac_canais, on="id_contrato", how="left")

    # A última consulta prévia é a única observável no momento da previsão.
    bureau_ref = filtra_por_data(
        bureau_locais, "id_contrato", "data_consulta", referencia
    )
    ultima_consulta = (
        bureau_ref.sort_values(["id_contrato", "data_consulta"])
        .groupby("id_contrato", as_index=False)
        .tail(1)[["id_contrato", "data_consulta", "score"]]
        .rename(columns={"score": "score_bureau"})
    )
    features = features.merge(ultima_consulta, on="id_contrato", how="left")
    features["tem_consulta_bureau"] = features["score_bureau"].notna().astype("int8")
    features["dias_desde_consulta_bureau"] = (
        features["data_referencia"] - features["data_consulta"]
    ).dt.days
    features = features.drop(columns="data_consulta")

    # Ausência de evento significa contagem zero. Médias, recências e razões inexistentes
    # permanecem ausentes para imputação aprendida apenas no conjunto de treino.
    colunas_contagem = [
        "qtd_compras", "qtd_compra_informatica", "qtd_compra_moveis",
        "qtd_compra_eletro", "qtd_compra_cama_mesa_banho",
        "qtd_parcelas_vencidas", "qtd_parcelas_pagas", "qtd_parcelas_pendentes",
        "qtd_atrasos_positivos", "qtd_sac_telefone", "qtd_sac_whatsapp",
        "qtd_sac_loja", "qtd_sac_site", "tem_consulta_bureau",
    ]
    features[colunas_contagem] = features[colunas_contagem].fillna(0).astype("int64")
    features[["qtd_contratos_cliente_pre_ref", "qtd_contratos_outros_pre_ref"]] = (
        features[["qtd_contratos_cliente_pre_ref", "qtd_contratos_outros_pre_ref"]]
        .fillna(0)
        .astype("int64")
    )

    if features["id_contrato"].duplicated().any() or len(features) != len(referencia):
        raise ValueError("A matriz final precisa ter exatamente uma linha por contrato")
    numericas = features.select_dtypes(include="number")
    if np.isinf(numericas.to_numpy()).any():
        raise ValueError("Features numéricas não podem conter infinito")

    return features

### 3.4 Auditoria de colunas

Antes de fechar o `features.csv`, passe **cada coluna** pelas três perguntas:

1. **Essa informação existiria no momento em que a previsão seria feita?**
2. **É uma variável protegida ou sensível?**
3. **Descreve comportamento, ou só descreve quem a pessoa é?**

❓ Monte a tabela: coluna | decisão (entra / sai) | justificativa. Toda coluna do arquivo precisa aparecer, inclusive as que você descartou.

Salve o resultado em `features.csv`.

In [6]:
features = construir_features(BASE_DIR, base_referencia)

colunas_excluidas = pd.DataFrame(
    [
        ["sexo", "sai", "atributo sensível; fora do escopo definido"],
        ["bairro", "sai", "proxy territorial potencialmente discriminatório; fora do escopo definido"],
        ["status_contrato", "sai", "situação no snapshot; não era conhecida na ligação preventiva"],
        ["data_snapshot", "sai", "constante na extração; usada apenas para definir a parcela elegível"],
        ["motivos_atraso", "sai", "não possui chave de contrato/cliente e pode ser pós-atraso"],
        ["qtd_negociacao_divida", "sai", "não há negociação de dívida anterior à referência; coluna seria constante"],
    ],
    columns=["coluna", "decisao", "justificativa"],
)

assert features["id_contrato"].nunique() == len(features) == len(contratos)
assert features["data_referencia"].notna().all()
assert not np.isinf(features.select_dtypes(include="number").to_numpy()).any()

features.to_csv("features.csv", index=False, encoding="utf-8")
print(f"features.csv salvo com {len(features):,} contratos e {features.shape[1]} colunas.")
display(features.head())
display(colunas_excluidas)

features.csv salvo com 3,000 contratos e 31 colunas.


,id_contrato,id_cliente,data_referencia,inadimplente_30d,n_parcelas,valor_parcela,renda_declarada_tratada,flag_renda_zero,canal_venda,tenure_dias,...,qtd_parcelas_pagas,qtd_parcelas_pendentes,qtd_atrasos_positivos,qtd_sac_telefone,qtd_sac_whatsapp,qtd_sac_loja,qtd_sac_site,score_bureau,tem_consulta_bureau,dias_desde_consulta_bureau
0,C00001,CLI00533,2026-01-22,0,10,101.50,6104.01,0,Loja física,263,...,8,0,4,0,1,0,0,600,1,263
1,C00003,CLI00116,2026-02-02,0,12,207.86,3944.55,0,Site,323,...,10,0,2,0,0,1,0,878,1,326
2,C00004,CLI00930,2025-09-05,1,10,151.67,1492.09,0,Loja física,293,...,9,0,5,0,0,0,0,689,1,296
3,C00005,CLI02675,2025-09-03,1,12,779.16,2230.13,0,Site,353,...,11,0,5,0,0,1,0,594,1,355
4,C00007,CLI00506,2026-01-29,1,12,164.53,1861.14,0,Loja física,263,...,8,0,2,0,0,0,0,518,1,265


,coluna,decisao,justificativa
0,sexo,sai,atributo sensível; fora do escopo definido
1,bairro,sai,proxy territorial potencialmente discriminatór...
2,status_contrato,sai,situação no snapshot; não era conhecida na lig...
3,data_snapshot,sai,constante na extração; usada apenas para defin...
4,motivos_atraso,sai,não possui chave de contrato/cliente e pode se...
5,qtd_negociacao_divida,sai,não há negociação de dívida anterior à referên...


---
## Fase 4 — Modelagem

- Split **estratificado**.
- Um **baseline simples** (árvore de decisão) e pelo menos mais um modelo.
- Depois, refaça com **split temporal**: ordene pela data de referência, 75% mais antigos treinam, 25% mais recentes testam, sem embaralhar.

❓ O resultado do split temporal foi diferente do aleatório? O que isso diz — e o que **não** diz?

In [ ]:
# TODO: split estratificado + baseline + segundo modelo


In [ ]:
# TODO: split temporal e comparacao


---
## Fase 5 — Avaliação

A equipe faz **80 ligações**. A avaliação acontece nesse corte, não no threshold de 0,50.

1. Ordene o conjunto de teste pela probabilidade prevista (`predict_proba`).
2. Corte nos 80 primeiros.
3. Calcule **precision** e **recall** nesse recorte.
4. Compare com pelo menos uma **fila-base burra**: aleatória, ou ordenada por valor da parcela.

❓ Sua fila é melhor que a fila-base? Quanto? Se a diferença for pequena, diga isso — é um resultado legítimo.

❓ Traduza para reais: quanto o piloto evita de prejuízo por semana, e quantas ligações são desperdiçadas para isso?

❓ Você prioriza **precision** ou **recall** neste caso? Justifique pelo negócio, não pela métrica.

In [ ]:
# TODO: precision@80, recall@80, fila-base, comparacao


In [ ]:
# TODO: importancia de variaveis - e cuidado ao interpretar


### Gerar a fila

Salve `fila_80.csv` com `id_contrato` e `probabilidade`, ordenado do maior risco para o menor.

In [ ]:
# TODO: gerar fila_80.csv


---
## Fechamento

Antes de enviar, confira o checklist da seção 9 do enunciado.

Faltam ainda dois arquivos que **não** são gerados aqui:
- `decisoes.md` — o que entrou, o que saiu, cada pedido do Ricardo respondido, e o model card.
- `apresentacao.pptx` — 3 slides, para o Ricardo, não para a banca.

> Se algum resultado ficou bom demais, pergunte: *que outra coisa poderia produzir esse mesmo número?*